# CLIP-space Concept Poisoning Proxy - Multi-target Benchmark

Notebook n?y ki?m tra Concept Poisoning proxy tr?n nhi?u target text kh?c nhau. M?i target ???c ch?y ??c l?p: v?i c?ng m?t subset ?nh, PGD t?o m?t b? ?nh poisoned ri?ng cho target ??.

- Dataset: Caltech-101 resized 224x224
- Model: OpenAI CLIP ViT-B/32
- Fixed epsilon: 0.03
- Metric ch?nh: target cosine gain, target rank improvement, top-1/top-5 after, PSNR, SSIM, L-infinity


## 1. Setup Colab / GitHub repo

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
REPO_DIR = Path("/content/adversarial-data-protection")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.datasets import get_caltech101
from src.evaluation import compute_linf, compute_psnr, compute_ssim
from src.pipeline import tensor_to_image
from src.techniques.concept_poisoning import (
    clip_target_metrics,
    clip_text_ranking,
    load_clip_model,
    poison_images,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 2. Paths and config

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    RESULTS_ROOT = f"{DRIVE_PROJECT_DIR}/results"
    DATA_ROOT = "/content/adp_data_cache"
    Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
    Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    print("Dataset will be read from local Colab disk. If missing, torchvision will download it into:", DATA_ROOT)

SEED = 42
IMG_SIZE = 224
SUBSET_SIZE = 300
BATCH_SIZE = 16
EPSILON = 0.03
PGD_STEPS = 10
TARGET_CONCEPTS = [
    "a photo of a dog",
    "a photo of a car",
    "a photo of a flower",
    "a photo of an airplane",
]
CANDIDATE_TEXTS = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car",
    "a photo of an airplane",
    "a photo of a flower",
    "a photo of a tree",
    "a photo of a chair",
    "a photo of a bird",
    "a photo of a face",
    "a photo of a building",
]
SAVE_SAMPLE_COUNT = 8

RUN_NAME = f"caltech101_multitarget_subset{SUBSET_SIZE}_eps{str(EPSILON).replace('.', 'p')}_pgd{PGD_STEPS}_seed{SEED}"
RUN_DIR = Path(RESULTS_ROOT) / "clip_concept_poisoning_proxy_multitarget" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
SAMPLE_DIR = RUN_DIR / "samples"
FIGURE_DIR = RUN_DIR / "figures"
TENSOR_DIR = RUN_DIR / "tensors"
for d in [TABLE_DIR, SAMPLE_DIR, FIGURE_DIR, TENSOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("RUN_DIR:", RUN_DIR)
print("TARGET_CONCEPTS:", TARGET_CONCEPTS)
print("EPSILON:", EPSILON)
print("PGD_STEPS:", PGD_STEPS)


## 3. Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)


## 4. Load dataset and CLIP

In [ ]:
data_loader, _unused_loader, num_classes = get_caltech101(
    root=DATA_ROOT,
    img_size=IMG_SIZE,
    subset_size=SUBSET_SIZE,
    batch_size=BATCH_SIZE,
    train_ratio=1.0,
    seed=SEED,
    download=True,
    num_workers=2,
)
print("num_classes:", num_classes)
print("num_images:", len(data_loader.dataset))
print("batches:", len(data_loader))

clip_model, _ = load_clip_model("ViT-B/32", DEVICE)
clip_model.eval()
print("CLIP model: ViT-B/32")


## 5. Metric helpers

In [ ]:
def rank_of_target(similarities, target_index):
    order = torch.argsort(similarities, dim=1, descending=True)
    ranks = []
    for row in order:
        ranks.append((row == target_index).nonzero(as_tuple=False).item() + 1)
    return torch.tensor(ranks)


def ranking_metrics(x_orig, x_protected, target_concept):
    sims_before = clip_text_ranking(clip_model, x_orig, CANDIDATE_TEXTS, DEVICE)
    sims_after = clip_text_ranking(clip_model, x_protected, CANDIDATE_TEXTS, DEVICE)
    target_idx = CANDIDATE_TEXTS.index(target_concept)
    rank_before = rank_of_target(sims_before, target_idx)
    rank_after = rank_of_target(sims_after, target_idx)
    return {
        "target_rank_before": rank_before,
        "target_rank_after": rank_after,
        "target_rank_improvement": rank_before - rank_after,
        "target_top1_before": (rank_before == 1).float(),
        "target_top1_after": (rank_after == 1).float(),
        "target_top5_before": (rank_before <= 5).float(),
        "target_top5_after": (rank_after <= 5).float(),
        "sims_before": sims_before,
        "sims_after": sims_after,
    }


def target_slug(text):
    return text.lower().replace(" ", "_").replace("/", "_")


def save_samples(x_orig, x_protected, target_concept, max_samples=8):
    target_dir = SAMPLE_DIR / target_slug(target_concept)
    target_dir.mkdir(parents=True, exist_ok=True)
    n = min(max_samples, x_orig.size(0))
    for i in range(n):
        tensor_to_image(x_orig[i].cpu()).save(target_dir / f"original_{i:03d}.png")
        tensor_to_image(x_protected[i].cpu()).save(target_dir / f"protected_{i:03d}.png")
        noise = ((x_protected[i].detach().cpu() - x_orig[i].detach().cpu()) * 10 + 0.5).clamp(0, 1)
        tensor_to_image(noise).save(target_dir / f"noise_x10_{i:03d}.png")


## 6. Multi-target benchmark

In [ ]:
all_summary = []

for target_concept in TARGET_CONCEPTS:
    print("=" * 80)
    print("Target:", target_concept)
    start_time = time.perf_counter()
    per_batch_rows = []
    first_orig = None
    first_protected = None
    first_sims_before = None
    first_sims_after = None

    for batch_idx, (x, _y) in enumerate(tqdm(data_loader, desc=target_concept)):
        x = x.to(DEVICE)
        x_protected = poison_images(
            clip_model,
            x,
            target_concept=target_concept,
            epsilon=EPSILON,
            pgd_steps=PGD_STEPS,
            pgd_alpha=max(EPSILON / max(PGD_STEPS, 1), 1 / 255),
            device=DEVICE,
        )
        target = clip_target_metrics(clip_model, x, x_protected, target_concept, DEVICE)
        ranks = ranking_metrics(x, x_protected, target_concept)
        row = {
            "batch_idx": batch_idx,
            "batch_size": x.size(0),
            "target_concept": target_concept,
            "epsilon": EPSILON,
            "target_cosine_before": round(float(target["target_cosine_before"].mean()), 4),
            "target_cosine_after": round(float(target["target_cosine_after"].mean()), 4),
            "target_cosine_gain": round(float(target["target_cosine_gain"].mean()), 4),
            "target_rank_before": round(float(ranks["target_rank_before"].float().mean()), 4),
            "target_rank_after": round(float(ranks["target_rank_after"].float().mean()), 4),
            "target_rank_improvement": round(float(ranks["target_rank_improvement"].float().mean()), 4),
            "target_top1_before": round(float(ranks["target_top1_before"].mean()), 4),
            "target_top1_after": round(float(ranks["target_top1_after"].mean()), 4),
            "target_top5_before": round(float(ranks["target_top5_before"].mean()), 4),
            "target_top5_after": round(float(ranks["target_top5_after"].mean()), 4),
            "psnr": compute_psnr(x.detach().cpu(), x_protected.detach().cpu()),
            "ssim": compute_ssim(x.detach().cpu(), x_protected.detach().cpu()),
            "linf": compute_linf(x.detach().cpu(), x_protected.detach().cpu()),
        }
        per_batch_rows.append(row)

        if first_orig is None:
            first_orig = x.detach().cpu()
            first_protected = x_protected.detach().cpu()
            first_sims_before = ranks["sims_before"].detach().cpu()
            first_sims_after = ranks["sims_after"].detach().cpu()

        del x, x_protected
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    per_batch_df = pd.DataFrame(per_batch_rows)
    per_batch_df.to_csv(TABLE_DIR / f"per_batch_{target_slug(target_concept)}.csv", index=False)

    weighted = per_batch_df.copy()
    total = weighted["batch_size"].sum()
    summary = {
        "technique": "clip_space_concept_poisoning_proxy",
        "dataset": "Caltech-101",
        "model": "OpenAI CLIP ViT-B/32",
        "subset_size": SUBSET_SIZE,
        "image_size": IMG_SIZE,
        "target_concept": target_concept,
        "candidate_text_count": len(CANDIDATE_TEXTS),
        "epsilon": EPSILON,
        "pgd_steps": PGD_STEPS,
        "target_cosine_before": round(float((weighted["target_cosine_before"] * weighted["batch_size"]).sum() / total), 4),
        "target_cosine_after": round(float((weighted["target_cosine_after"] * weighted["batch_size"]).sum() / total), 4),
        "target_cosine_gain": round(float((weighted["target_cosine_gain"] * weighted["batch_size"]).sum() / total), 4),
        "target_rank_before": round(float((weighted["target_rank_before"] * weighted["batch_size"]).sum() / total), 4),
        "target_rank_after": round(float((weighted["target_rank_after"] * weighted["batch_size"]).sum() / total), 4),
        "target_rank_improvement": round(float((weighted["target_rank_improvement"] * weighted["batch_size"]).sum() / total), 4),
        "target_top1_before": round(float((weighted["target_top1_before"] * weighted["batch_size"]).sum() / total), 4),
        "target_top1_after": round(float((weighted["target_top1_after"] * weighted["batch_size"]).sum() / total), 4),
        "target_top5_before": round(float((weighted["target_top5_before"] * weighted["batch_size"]).sum() / total), 4),
        "target_top5_after": round(float((weighted["target_top5_after"] * weighted["batch_size"]).sum() / total), 4),
        "psnr": round(float((weighted["psnr"] * weighted["batch_size"]).sum() / total), 4),
        "ssim": round(float((weighted["ssim"] * weighted["batch_size"]).sum() / total), 4),
        "linf_max": round(float(weighted["linf"].max()), 4),
        "runtime_seconds": round(time.perf_counter() - start_time, 2),
        "run_name": RUN_NAME,
        "run_dir": str(RUN_DIR),
    }
    all_summary.append(summary)
    save_samples(first_orig, first_protected, target_concept, SAVE_SAMPLE_COUNT)
    torch.save(
        {
            "x_orig": first_orig,
            "x_protected": first_protected,
            "candidate_texts": CANDIDATE_TEXTS,
            "target_concept": target_concept,
            "sims_before": first_sims_before,
            "sims_after": first_sims_after,
        },
        TENSOR_DIR / f"sample_batch_{target_slug(target_concept)}.pt",
    )
    print(summary)

summary_df = pd.DataFrame(all_summary)
summary_path = TABLE_DIR / "clip_concept_poisoning_multitarget_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)
summary_df


## 7. Plot multi-target comparison

In [ ]:
summary_df = pd.read_csv(TABLE_DIR / "clip_concept_poisoning_multitarget_summary.csv")
labels = summary_df["target_concept"].str.replace("a photo of a ", "", regex=False)
x = np.arange(len(labels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(x - width / 2, summary_df["target_cosine_gain"], width, label="target cosine gain")
ax1.bar(x + width / 2, summary_df["target_top1_after"], width, label="target top-1 after")
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=20, ha="right")
ax1.set_ylabel("CLIP behavior metric")
ax1.grid(True, axis="y", alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(x, summary_df["psnr"], marker="o", color="green", label="PSNR")
ax2.plot(x, summary_df["ssim"], marker="o", color="purple", label="SSIM")
ax2.set_ylabel("image quality")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "concept_poisoning_multitarget_comparison.png", dpi=160)
plt.show()
print("Saved:", FIGURE_DIR / "concept_poisoning_multitarget_comparison.png")


## 8. Interpretation notes

In [ ]:
print("Each target is optimized independently and produces its own protected images.")
print("This benchmark tests whether the same CLIP-space PGD method works across multiple target concepts.")
print("It does not mean one single poisoned image set satisfies all targets at once.")
print("RUN_DIR:", RUN_DIR)
